In [1]:
import os
from dotenv import load_dotenv

load_dotenv()

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

from langchain.chat_models import init_chat_model

model = init_chat_model(
  model="llama-3.1-8b-instant",
  model_provider="GROQ"
)

c:\Users\anand\Desktop\GenAI - Krish Naik\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver

def read_email(email_id: str) -> str:
  """Mock function to read email by emailID"""
  return f"Email content for ID: {email_id}"


def send_email(recipient: str, subject: str, body: str) -> str:
  """Mock function to send an email"""
  return f"Email sent to {recipient} with subject {subject}"

In [3]:
agent = create_agent(
  model=model,
  tools=[read_email, send_email],
  checkpointer=InMemorySaver(),
  middleware=[
    HumanInTheLoopMiddleware(
      interrupt_on={
        "send_email":{
          "allowed_decisions":["approve", "edit", "reject"]
        },
        "read_email": False
      }
    )
  ]
)

In [4]:
from langchain_core.messages import HumanMessage

config = {"configurable": {"thread_id": "test"}}

response = agent.invoke(
  {
    "messages": HumanMessage(content="Send email to johndoe@test.com with subject 'Hello' and body 'How are you?'")
  },config=config
)

In [5]:
response

{'messages': [HumanMessage(content="Send email to johndoe@test.com with subject 'Hello' and body 'How are you?'", additional_kwargs={}, response_metadata={}, id='c514ee97-16a4-4037-95df-34caa1f729f6'),
  AIMessage(content='', additional_kwargs={'tool_calls': [{'id': '3cxt3g4p7', 'function': {'arguments': '{"body":"How are you?","recipient":"johndoe@test.com","subject":"Hello"}', 'name': 'send_email'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 34, 'prompt_tokens': 306, 'total_tokens': 340, 'completion_time': 0.033512266, 'completion_tokens_details': None, 'prompt_time': 0.028256818, 'prompt_tokens_details': None, 'queue_time': 0.045679101, 'total_time': 0.061769084}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019ec254-e025-71c3-8aa8-0fe4f537d164-0', tool_calls=[{'name': 'send_email', 'args': {'body': '

In [7]:
# Human Approval
from langgraph.types import Command

if "__interrupt__" in response:
  print("Paused Approving...")
  
  result = agent.invoke(
    Command(
      resume={
        "decisions":[
          {"type": "approve"}
        ]
      }
    ),
    config=config
  )
  
  print(f"{result['messages'][-1].content}")

Paused Approving...

